# Hands-On 17: NetDiffusion on Colab

This notebook runs the [NetDiffusion](https://github.com/noise-lab/NetDiffusion) pipeline end to end on Google Colab: nPrint packet captures become images, a LoRA adapter is fine-tuned on Stable Diffusion 3, ControlNet guides generation, and a post-processing pass turns the generated images back into replayable PCAPs.

**Before you start**

1. **Runtime → Change runtime type → GPU** (a T4 is enough for the miniature training run; an A100 is faster if you have one).
2. **Stable Diffusion 3 is gated.** Create a free Hugging Face account, open the [SD3 Medium model page](https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers), fill in the form and accept the terms (access is usually granted immediately). Then make a **read** token under *Settings → Access Tokens*; the login cell below will ask for it.
3. Budget on a T4: installs about 3 minutes, dataset download under a minute, embeddings about 2 minutes, the one-step LoRA run about 4 minutes, generation about 8 minutes, post-processing under a minute. **Runtime → Run all** works once the Hugging Face token is in Colab secrets (see the authentication cell).

The cells below are the upstream notebook's, in order, with the repository cloned first so the `scripts/` paths resolve on Colab. Read each markdown cell before running the code under it; the point of the hands-on is to see the four steps of the pipeline, not to wait on the GPU.


## Step 0: Get the code

Clone the NetDiffusion repository and work inside it, so the `./scripts/...` paths in the rest of the notebook resolve.

In [ ]:
!git clone -q https://github.com/noise-lab/NetDiffusion.git
%cd NetDiffusion
!ls scripts

## Install Dependencies

The versions are pinned. The NetDiffusion scripts were written against `diffusers` 0.30 to 0.33 and `transformers` 4.x; `transformers` 5 removed the `load_in_8bit` argument the embedding script uses, and `huggingface_hub` 1.0 removed the `huggingface-cli` command. Colab's own `torch` and `torchvision` are left alone.

- `diffusers`, `transformers`, `accelerate`, `peft`, `bitsandbytes`: Stable Diffusion 3, the LoRA training loop, and 8-bit Adam
- `sentencepiece`, `controlnet_aux`, `opencv-python`: the T5 tokenizer and the Canny control image
- `scapy`, `pyarrow`, `pandas`, `gdown`: PCAP checksums, the embeddings parquet, and the dataset download

About two minutes.

In [ ]:
!pip install -q "diffusers==0.33.1" "transformers==4.51.3" "huggingface_hub<1.0" \
    accelerate peft bitsandbytes sentencepiece controlnet_aux opencv-python scapy pyarrow pandas gdown
import diffusers, transformers, huggingface_hub
print("diffusers", diffusers.__version__, "transformers", transformers.__version__, "huggingface_hub", huggingface_hub.__version__)

### Install nPrint

The last step of the pipeline (`mass_reconstruction.py`) shells out to the `nprint` binary to turn the generated nPrint matrices back into PCAPs and to re-encode them. Colab does not have it, so build it from the 1.2.1 release. The `-include cstdint` flag is needed because the release predates GCC 13's stricter headers. About a minute.

In [ ]:
%%bash
set -e
apt-get install -qq -y libpcap-dev > /dev/null
cd /tmp
curl -sL https://github.com/nprint/nprint/releases/download/v1.2.1/nprint-1.2.1.tar.gz | tar xz
cd nprint-1.2.1
./configure -q > /dev/null
make -s -j4 CXXFLAGS='-g -O2 -include cstdint' > /dev/null
make -s install > /dev/null
nprint --version

## Hugging Face Authentication

Stable Diffusion 3 is gated. Once per account: open the [SD3 Medium model page](https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers), fill in the form and accept the terms, then create a **read** token under *Settings → Access Tokens*.

Two ways to hand the token to this runtime:

1. **Colab secrets (recommended).** Click the key icon in the left sidebar, add a secret named `HF_TOKEN` with the token as its value, and turn on notebook access. The cell below then finds it with no prompt, every time you reopen the notebook.
2. **Paste it.** If there is no secret, the cell opens a login box; paste the token there.

(The old `huggingface-cli login` command was removed from `huggingface_hub` 1.0 and is not used here.)

In [ ]:
from huggingface_hub import login, whoami

try:
    print("Already logged in as", whoami()["name"])
except Exception:
    login()  # reads the HF_TOKEN Colab secret if there is one, otherwise prompts
    print("Logged in as", whoami()["name"])

## Download Example Dataset

Alternatively, create your own dataset:
1. Install nPrint in your environment (see https://nprint.github.io/nprint/)
2. Collect PCAPs belonging to the same service/application you plan to model
3. Convert each PCAP to an nPrint file with the following command:
   ```
   nprint -F -1 -P {pcap} -4 -i -6 -t -u -p 0 -c 1024 -W {output_file}
   ```
4. Place all resulting nPrint files in a folder named "nprint_traffic"

In [ ]:
!gdown 1vvneSH0a1WZFPHTafKusOUjNhg7oQioq -O preprocessed_dataset.zip
!unzip -q -o preprocessed_dataset.zip
!mkdir -p nprint_traffic
!mv amazon_nprint_traffic/* nprint_traffic/
!ls nprint_traffic | head -3; ls nprint_traffic | wc -l

## Convert nPrint to PNG Images

This script transforms `.nprint` files—tabular feature representations of network packets—into fixed-size PNG images that can be used as input for SD fine-tuning.

**Input:** `.nprint` files generated from packet captures (PCAPs) using the nPrint tool. Each file is a CSV-like matrix where each row represents a single packet and columns correspond to extracted features.

**Preprocessing:**
- Drops IP address-related columns to avoid injecting identifiable or non-generalizable information into the model
- Maps integer values in the remaining columns to RGBA color tuples to visualize numeric features as colored pixels

**Padding:** Pads each image to a uniform height (default 1024) using a solid background to ensure model input consistency across varying packet counts.

**Output:** Saves a PNG file for each `.nprint` file, preserving the packet structure as a vertically stacked color-coded image (rows = packets, cols = features).

In [ ]:
!python ./scripts/nprint_to_png.py -i ./nprint_traffic/ -o ./nprint_traffic_images

## Compute Embeddings

Generates text prompt embeddings via a Stable Diffusion 3 pipeline and T5 text encoder. Maps each local PNG image to a unique SHA-256 hash and associates it with the computed embeddings. Stores the resulting image-hash-to-embedding data in a .parquet file for further processing.

Here we are using the default instance prompt "pixelated network data for type-0 application traffic". You can configure this by referring to the compute_embeddings.py script for details on other supported arguments.

In [ ]:
!python ./scripts/compute_embeddings.py

## Train LoRA Adapter on Stable Diffusion 3 (Miniature Setup)

This command launches training using `accelerate` with DreamBooth-style LoRA tuning, optimized for quick experimentation or demo runs.

**⚠️ Current configuration uses:**
- Only 1 training step
- Small batch size
- No warmup
- High learning rate

**Intended only for testing or verifying training scripts**, NOT for quality results. For actual training, increase `max_train_steps`, adjust learning rate, and consider enabling full validation and saving checkpoints.

On a T4 this takes about four minutes, almost all of it loading the model. It writes `trained-sd3-lora-miniature/pytorch_lora_weights.safetensors`.

In [ ]:
!accelerate launch ./scripts/train_dreambooth_lora_sd3_miniature.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-3-medium-diffusers"  \
  --instance_data_dir="nprint_traffic_images" \
  --data_df_path="sample_embeddings.parquet" \
  --output_dir="trained-sd3-lora-miniature" \
  --mixed_precision="fp16" \
  --instance_prompt="pixelated network data for type-0 application traffic" \
  --train_batch_size=2 \
  --gradient_accumulation_steps=1 --gradient_checkpointing \
  --use_8bit_adam \
  --learning_rate=5e-5 \
  --report_to="tensorboard" \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=1 \
  --seed="0"

## Inference Pipeline for SD3 + ControlNet + LoRA

This cell demonstrates the full generation process:
- Loads Stable Diffusion 3 Medium base model
- Loads ControlNet (Canny edge-based guidance)
- Loads LoRA weights fine-tuned on pixelated network traffic
- Applies edge-conditioned generation using a sample input

In [ ]:
import os
import torch
import cv2
from PIL import Image
from diffusers import StableDiffusion3ControlNetPipeline, SD3ControlNetModel
from diffusers.utils import load_image

# Make sure our output folder exists
os.makedirs("generated_traffic_images", exist_ok=True)

In [ ]:
# Base SD 3.0 model
base_model_path = "stabilityai/stable-diffusion-3-medium-diffusers"

# Canny-based ControlNet
controlnet_path = "InstantX/SD3-Controlnet-Canny"

# Load the ControlNet and pipeline in half precision
controlnet = SD3ControlNetModel.from_pretrained(controlnet_path, torch_dtype=torch.float16)
pipe = StableDiffusion3ControlNetPipeline.from_pretrained(
    base_model_path,
    controlnet=controlnet,
    torch_dtype=torch.float16,
)

# Load the LoRA weights trained above
pipe.load_lora_weights("trained-sd3-lora-miniature")

# The full pipeline (three text encoders, the transformer, the VAE) does not fit on a T4 at once;
# sequential offload moves each module to the GPU only while it runs
pipe.enable_sequential_cpu_offload()

In [ ]:
# Convert original control image to Canny edges via OpenCV
orig_path = "./scripts/traffic_conditioning_image.png"
orig_bgr = cv2.imread(orig_path)
if orig_bgr is None:
    raise ValueError(f"Could not load file: {orig_path}")

# Convert to grayscale
gray = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2GRAY)

# Generate Canny edge map (tweak thresholds as needed)
edges = cv2.Canny(gray, 100, 200)

# Convert single-channel edge map to 3-channel RGB
edges_rgb = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)

# Convert to PIL for use in ControlNet pipeline
control_image = Image.fromarray(edges_rgb)
orig_width, orig_height = control_image.size
target_width = 1024
if orig_width > target_width:
    # (left, upper, right, lower)
    control_image = control_image.crop((0, 0, target_width, orig_height))
    
print("Displaying Canny control image:")
display(control_image)

Now generate. Twenty denoising steps at 1024 rows by 1088 columns (one row per packet, one column per nPrint bit) take about eight minutes on a T4 with sequential offload (about 22 seconds a step); the progress bar is per step. The LoRA adapter was trained for one step, so the output is mostly the base model guided by the Canny edges of the conditioning image; a real run trains for thousands of steps.

In [ ]:
prompt = "pixelated network data for type-0 application traffic"
generator = torch.manual_seed(0)  # reproducibility

image = pipe(
    prompt=prompt,
    num_inference_steps=20,
    generator=generator,
    height=1024,
    width=1088,  # the nPrint layout has 1088 columns; image_to_nprint.py needs exactly that width
    control_image=control_image,
    controlnet_conditioning_scale=0.5,  # increase to adhere more strongly to the edges
).images[0]

output_path = os.path.join("generated_traffic_images", "generated_traffic.png")
image.save(output_path)
print(f"Generated image saved to: {output_path}")
display(image)

## Post-Generation Processing Pipeline

This cell performs a 3-stage transformation of the generated PNG images:

1. **Color Augmentation:** Applies standardized color shifts to improve nprint reconstruction accuracy
2. **Image-to-nPrint Conversion:** Converts augmented images back into nPrint-compatible feature format using a reference file to maintain consistent structure
3. **Heuristic Correction & PCAP Reconstruction:** Reconstructs valid and replayable PCAP files from the diffusion-generated nPrint representation

This pipeline enables turning synthetic traffic images back into replayable network traffic for evaluation or simulation.

In [ ]:
# Step 1: Color Augmentation
!python ./scripts/color_processor.py \
  --input_dir="./generated_traffic_images" \
  --output_dir="./color_corrected_generated_traffic_images"

In [ ]:
# Step 2: Image-to-nPrint Conversion
!python ./scripts/image_to_nprint.py \
  --org_nprint ./scripts/column_example.nprint \
  --input_dir ./color_corrected_generated_traffic_images \
  --output_dir ./generated_nprint

In [ ]:
# Step 3: Heuristic Correction & PCAP Reconstruction
!python ./scripts/mass_reconstruction.py \
  --input_dir ./generated_nprint \
  --output_pcap_dir ./replayable_generated_pcaps \
  --output_nprint_dir ./replayable_generated_nprints \
  --formatted_nprint_path ./scripts/correct_format.nprint

Final PCAP files are stored in `replayable_generated_pcaps`

## Questions to answer in your notes

1. Look at one of the PNGs in `nprint_traffic_images/` next to its `.nprint` file. Which packet fields map to which pixel rows, and why does a packet capture look like an image at all?
2. What does the LoRA adapter change about Stable Diffusion, and what does ControlNet change? Which of the two could you drop and still get traffic-shaped images?
3. Open a generated PCAP in Wireshark or `tshark`. Which protocol-compliance fixes did `mass_reconstruction.py` have to make, and which problems does it leave (sequence numbers, handshake order, DNS before connect)?
4. The training set is one application's traffic. What would you need to generate traffic for a second application, and what would you need to generate a mixed session of both?
